# Direct Path Patching in TransformerLens

This notebook demonstrates **direct path patching** — a causal intervention technique for mechanistic interpretability.

## What is path patching?

Standard **activation patching** replaces the full residual stream at a layer with activations from a different ("clean") run. This tells you: *does this layer's residual stream matter for the output?*

**Direct path patching** is more targeted: it patches only the contribution of a specific component (e.g., a single attention head) to a specific downstream position, without affecting other paths. This lets you isolate *which component, sending to which position, causes the output difference*.

### The difference

| Technique | What you patch | What you learn |
|---|---|---|
| Activation patching | Full residual stream at layer L, position p | Does the state at (L, p) matter? |
| Direct path patching | Output of component C → position p only | Does C's direct contribution to p matter? |

## Setup

In [ ]:
# Install if needed
# !pip install transformer_lens

In [ ]:
import torch
from transformer_lens import HookedTransformer
import einops

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
print(f"Loaded GPT-2: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## Part 1: Basic Activation Patching

Before doing direct path patching, let's establish the baseline: full residual stream patching.

We use a **clean prompt** and a **corrupted prompt** that differ in one key token.

In [ ]:
# Classic IOI-style setup: two prompts differing in one name
clean_prompt   = "When Mary and John went to the store, John gave a bag to"
corrupt_prompt = "When Mary and John went to the store, Mary gave a bag to"

clean_tokens   = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

# The clean answer is " Mary", the corrupted answer is " John"
answer_token = model.to_single_token(" Mary")
wrong_token  = model.to_single_token(" John")

print(f"Clean tokens shape:   {clean_tokens.shape}")
print(f"Correct answer token: {answer_token} = '{model.to_string(answer_token)}'")
print(f"Wrong answer token:   {wrong_token}  = '{model.to_string(wrong_token)}'")

In [ ]:
def logit_diff(logits, answer_tok=answer_token, wrong_tok=wrong_token):
    """Difference in logits between the correct and incorrect answer tokens."""
    return logits[0, -1, answer_tok] - logits[0, -1, wrong_tok]

# Baseline measurements
with torch.no_grad():
    clean_logits,   clean_cache   = model.run_with_cache(clean_tokens)
    corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_tokens)

clean_diff   = logit_diff(clean_logits).item()
corrupt_diff = logit_diff(corrupt_logits).item()

print(f"Clean logit diff   (higher = model says Mary): {clean_diff:+.3f}")
print(f"Corrupted logit diff:                         {corrupt_diff:+.3f}")
print(f"\nTotal effect to explain: {clean_diff - corrupt_diff:.3f}")

In [ ]:
def patch_residual_stream(layer, position):
    """
    Patch the residual stream at (layer, position) from the clean cache
    into a corrupted forward pass. Returns the normalized logit diff.
    """
    hook_name = f"blocks.{layer}.hook_resid_post"
    clean_act = clean_cache[hook_name][:, position, :].clone()

    def patch_hook(resid, hook):
        resid[:, position, :] = clean_act
        return resid

    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            corrupt_tokens,
            fwd_hooks=[(hook_name, patch_hook)]
        )

    patched_diff = logit_diff(patched_logits).item()
    # Normalized: 0 = no recovery, 1 = full recovery
    return (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)


# Scan all layers at the final token position
final_pos = clean_tokens.shape[1] - 1
print(f"Patching at final token position ({final_pos}):\n")
print(f"{'Layer':<8} {'Normalized recovery':>20}")
print("-" * 30)

for layer in range(model.cfg.n_layers):
    recovery = patch_residual_stream(layer, final_pos)
    bar = "█" * int(abs(recovery) * 20)
    print(f"  L{layer:<5} {recovery:>+.3f}  {bar}")

## Part 2: Direct Path Patching

Full residual stream patching tells us *which layer* matters. But it includes contributions from all upstream components. Direct path patching isolates the contribution of a **single attention head** to the final position.

### How it works

An attention head's output is added to the residual stream at every position. To patch the **direct path** from head `(layer, head)` to the final position:

1. Run the **clean** forward pass with cache
2. Start a **corrupted** forward pass
3. At the attention output hook for `(layer, head)`, replace only that head's output at the target position with the clean value
4. Let the rest of the forward pass continue normally

This isolates the head's direct write to the residual stream — without changing what the head attends to in earlier layers.

In [ ]:
def patch_head_output(layer, head, position):
    """
    Patch the direct output of attention head (layer, head) at `position`
    from the clean cache into a corrupted forward pass.
    Returns normalized logit diff recovery.
    """
    hook_name = f"blocks.{layer}.attn.hook_result"
    # hook_result shape: [batch, seq, n_heads, d_model]
    clean_head_out = clean_cache[hook_name][:, position, head, :].clone()

    def patch_hook(result, hook):
        result[:, position, head, :] = clean_head_out
        return result

    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            corrupt_tokens,
            fwd_hooks=[(hook_name, patch_hook)]
        )

    patched_diff = logit_diff(patched_logits).item()
    return (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)


# Scan all heads at the final token position
print(f"Direct path patching: head outputs → final position ({final_pos})")
print(f"(Normalized recovery: 0=no effect, 1=full recovery)\n")

results = {}
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        recovery = patch_head_output(layer, head, final_pos)
        results[(layer, head)] = recovery

# Show top 10 most important heads
sorted_heads = sorted(results.items(), key=lambda x: abs(x[1]), reverse=True)
print(f"{'Head':<12} {'Recovery':>12}")
print("-" * 26)
for (layer, head), recovery in sorted_heads[:10]:
    bar = "█" * int(abs(recovery) * 30)
    print(f"  L{layer}H{head:<6} {recovery:>+.3f}  {bar}")

## Part 3: Visualize the Path Patching Results

Plot a heatmap of direct path recovery across all (layer, head) pairs.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    grid = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
    for (layer, head), recovery in results.items():
        grid[layer, head] = recovery

    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(grid, cmap="RdBu", vmin=-0.5, vmax=0.5, aspect="auto")
    ax.set_xlabel("Head", fontsize=12)
    ax.set_ylabel("Layer", fontsize=12)
    ax.set_title(
        "Direct Path Patching Recovery\n"
        "(Clean → Corrupted, patching head output at final token position)",
        fontsize=13
    )
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    plt.colorbar(im, ax=ax, label="Normalized logit diff recovery")
    plt.tight_layout()
    plt.show()
    print("Heads with high recovery (red) have strong direct causal influence on the output.")
except ImportError:
    print("matplotlib not installed — skipping plot. Top heads shown in table above.")

## Part 4: Why Direct Path Patching Matters

### Difference from full activation patching

Full residual stream patching at layer L includes contributions from *all* heads in layers 0..L plus all MLPs. It's a coarse measure.

Direct path patching isolates a single head's write, making it much more precise for identifying specific circuit components.

### Caution: the "no mediation" assumption

Direct path patching measures the **direct** causal path: `head output → final position → logits`. It does NOT capture:
- Indirect paths where head A's output is read by head B, which then writes to the final position
- Effects mediated through MLPs

For full circuit analysis, combine direct path patching with attention pattern analysis (`hook_attn`) to trace indirect paths.

### Connection to the circuit discovery literature

Direct path patching is the core operation in:
- Wang et al. (2022): *Interpretability in the Wild* (IOI circuit)
- Conmy et al. (2023): *ACDC* (automated circuit discovery)
- Marks et al. (2024): *Sparse Feature Circuits*

### Limitations and robustness

**Important:** High path patching recovery at a fixed position does not guarantee that the head is causally necessary for the behavior in all input distributions. Heads that appear causal on one prompt distribution may have their contribution "read" by the model differently on adversarially constructed inputs.

Always validate circuit findings with:
1. Causal ablations (zero-ablate or mean-ablate the head)
2. Multiple prompt distributions
3. Cross-model replication

## Summary

| Operation | TransformerLens hook | What it patches |
|---|---|---|
| Residual stream patching | `blocks.{L}.hook_resid_post` | Full residual stream at layer L |
| Direct head output patching | `blocks.{L}.attn.hook_result` | Single head's write to residual stream |
| Attention pattern patching | `blocks.{L}.attn.hook_attn` | Which tokens a head attends to |
| MLP output patching | `blocks.{L}.hook_mlp_out` | MLP contribution to residual stream |

Each hook corresponds to a different granularity of causal intervention. Start coarse (residual stream), then refine (head output) to identify the minimal circuit responsible for a behavior.